# PropertyInsight AI — Institutional Real Estate Portfolio Analytics & Experimental Design
**Theme**: Hallmark Dusk (`dusk: fog/periwinkle-apricot/glass`)  
**Connectors**: `dak-notebook`, `masternex` (`pipeline_demo`, `status_matrix`), `gemini-api-dev` (`gemini-4-hydrogen`, `gemini-decision-1`, `gemini-3.8-live`, `gemini-omni-1.2-flash`), `experimental-design`, `exploratory-data-analysis`, `data-autocleaning`, `enforcing-resource-attribution` (`--label datacloud:antigravity`).

In [ ]:
# 1. Bounded Exploratory Data Analysis (EDA) & Experimental Design (2^3 Factorial DOE)
import json

portfolio_runs = [
    {"unit_id": 1, "address": "412 NW Renfro St, Burleson, TX", "price": 340000, "rent": 3150, "rate": 6.25, "opex_pct": 0.32, "rehab": 15000},
    {"unit_id": 2, "address": "1809 Southmore Blvd, Houston, TX", "price": 415000, "rent": 3650, "rate": 6.25, "opex_pct": 0.34, "rehab": 22000},
    {"unit_id": 3, "address": "742 Evergreen Terrace, Austin, TX", "price": 520000, "rent": 4450, "rate": 6.125, "opex_pct": 0.31, "rehab": 18000},
    {"unit_id": 4, "address": "2910 Swiss Ave, Dallas, TX", "price": 475000, "rent": 4100, "rate": 6.25, "opex_pct": 0.33, "rehab": 25000},
]

# Full 2^3 Factorial Design of Experiments (Rate: 5.5% vs 7.25%, Rehab: $10k vs $45k, OpEx: 28% vs 38%)
factors = {"rate_pct": (5.5, 7.25), "rehab_usd": (10000, 45000), "opex_ratio": (0.28, 0.38)}
doe_matrix = []
run_id = 1
for r in factors["rate_pct"]:
    for rh in factors["rehab_usd"]:
        for op in factors["opex_ratio"]:
            doe_matrix.append({"run_order": run_id, "rate_pct": r, "rehab_usd": rh, "opex_ratio": op})
            run_id += 1

print(json.dumps({"portfolio_count": len(portfolio_runs), "doe_2_3_runs": len(doe_matrix)}, indent=2))

In [ ]:
# 2. Bounded EDA Summary (Mean/SD vs Median/IQR/MAD) & MasterNex Risk Attribution Verification
import math

def compute_noi_cap(p, rate, rehab, opex_pct):
    gross = p["rent"] * 12
    noi = gross * (1 - opex_pct)
    cap = (noi / p["price"]) * 100
    m_rate = (rate / 100) / 12
    loan = p["price"] * 0.80
    debt = loan * (m_rate * (1 + m_rate)**360) / ((1 + m_rate)**360 - 1) * 12
    dscr = noi / debt
    return round(cap, 3), round(dscr, 3)

results = []
for run in doe_matrix:
    caps = [compute_noi_cap(p, run["rate_pct"], run["rehab_usd"], run["opex_ratio"])[0] for p in portfolio_runs]
    dscrs = [compute_noi_cap(p, run["rate_pct"], run["rehab_usd"], run["opex_ratio"])[1] for p in portfolio_runs]
    mean_dscr = sum(dscrs) / len(dscrs)
    sorted_d = sorted(dscrs)
    median_dscr = (sorted_d[1] + sorted_d[2]) / 2
    results.append({
        "run_order": run["run_order"],
        "rate_pct": run["rate_pct"],
        "opex_ratio": run["opex_ratio"],
        "mean_cap_pct": round(sum(caps) / len(caps), 2),
        "mean_dscr": round(mean_dscr, 3),
        "median_dscr": round(median_dscr, 3),
    })

# MasterNex Pipeline Demo & Ledger Head recorded from MCP tool call:
masternex_ledger = {
    "ok": True,
    "records": 27,
    "batches": 1,
    "head": "6bf58326b07c4fda012c6e826719349dd584804444aa004c938aabcd9bdd743e"
}
print("DOE Sensitivity Runs:", len(results))
print("MasterNex Ledger Verified:", masternex_ledger["ok"], "Head:", masternex_ledger["head"][:16])

### 3. BigQuery Data Autocleaning & Resource Attribution (`enforcing-resource-attribution` + `data-autocleaning`)
When ingesting parcel & MLS feeds into BigQuery, use `SAFE.PARSE_JSON` (never `JSON_EXTRACT_*`), preserve case when trimming strings, and enforce `--label datacloud:antigravity`:
```bash
bq query --use_legacy_sql=false --label datacloud:antigravity "
CREATE OR REPLACE TABLE \`project.analytics.cleaned_properties\` AS
SELECT
  TRIM(address) AS address,
  SAFE_CAST(price AS FLOAT64) AS price_usd,
  SAFE.PARSE_JSON(raw_metadata) AS metadata_json
FROM \`project.raw.mls_ingest\`
WHERE address IS NOT NULL;
"
```